In [2]:
import pandas as pd

In [3]:
price_dataset = pd.read_csv("../data/raw/entso-e-prices/Spain.csv")
price_dataset["datetime"] = pd.to_datetime(price_dataset["Datetime (UTC)"])

In [4]:
price_dataset.head()

,Country,ISO3 Code,Datetime (UTC),Datetime (Local),Price (EUR/MWhe),datetime
0,Spain,ESP,2015-01-01 00:00:00,2015-01-01 01:00:00,48.10,2015-01-01 00:00:00
1,Spain,ESP,2015-01-01 01:00:00,2015-01-01 02:00:00,47.33,2015-01-01 01:00:00
2,Spain,ESP,2015-01-01 02:00:00,2015-01-01 03:00:00,42.27,2015-01-01 02:00:00
3,Spain,ESP,2015-01-01 03:00:00,2015-01-01 04:00:00,38.41,2015-01-01 03:00:00
4,Spain,ESP,2015-01-01 04:00:00,2015-01-01 05:00:00,35.72,2015-01-01 04:00:00


In [5]:
price_dataset = price_dataset.drop(columns=["Country", "ISO3 Code", 'Datetime (Local)', 'Datetime (UTC)'])

In [6]:
price_dataset.head()

,Price (EUR/MWhe),datetime
0,48.10,2015-01-01 00:00:00
1,47.33,2015-01-01 01:00:00
2,42.27,2015-01-01 02:00:00
3,38.41,2015-01-01 03:00:00
4,35.72,2015-01-01 04:00:00


In [7]:
solar_dataset = pd.read_csv("../data/raw/ES/solar-raw.csv")

In [8]:
solar_dataset.head()

,Unnamed: 0,solar_generation_MW
0,2022-01-01 00:00:00+00:00,75.0
1,2022-01-01 01:00:00+00:00,75.0
2,2022-01-01 02:00:00+00:00,75.0
3,2022-01-01 03:00:00+00:00,75.0
4,2022-01-01 04:00:00+00:00,75.0


In [9]:
solar_dataset.columns = ["datetime", "solar_generation_MW"]

In [10]:
solar_dataset.head()

,datetime,solar_generation_MW
0,2022-01-01 00:00:00+00:00,75.0
1,2022-01-01 01:00:00+00:00,75.0
2,2022-01-01 02:00:00+00:00,75.0
3,2022-01-01 03:00:00+00:00,75.0
4,2022-01-01 04:00:00+00:00,75.0


In [11]:
solar_dataset["datetime"] = pd.to_datetime(solar_dataset["datetime"], utc=True).dt.tz_localize(None)

In [12]:
solar_dataset.isna().sum()

datetime               0
solar_generation_MW    0
dtype: int64

In [13]:
meteo_dataset = pd.read_csv("../data/raw/meteo-raw-spain.csv")

In [14]:
meteo_dataset.head()

,time,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation
0,2022-01-01T00:00,8.7,9.3,103,0,0.0,0.0
1,2022-01-01T01:00,10.2,8.6,112,26,0.0,0.0
2,2022-01-01T02:00,9.0,10.3,115,0,0.0,0.0
3,2022-01-01T03:00,11.7,7.1,105,0,0.0,0.0
4,2022-01-01T04:00,13.3,4.7,99,0,0.0,0.0


In [15]:
meteo_dataset["datetime"] = pd.to_datetime(meteo_dataset["time"])

In [16]:
meteo_dataset.columns

Index(['time', 'temperature_2m', 'wind_speed_10m', 'wind_direction_10m',
       'cloud_cover', 'shortwave_radiation', 'precipitation', 'datetime'],
      dtype='object')

In [17]:
meteo_dataset = meteo_dataset.drop(columns=["time"])

In [18]:
meteo_dataset.head()

,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation,datetime
0,8.7,9.3,103,0,0.0,0.0,2022-01-01 00:00:00
1,10.2,8.6,112,26,0.0,0.0,2022-01-01 01:00:00
2,9.0,10.3,115,0,0.0,0.0,2022-01-01 02:00:00
3,11.7,7.1,105,0,0.0,0.0,2022-01-01 03:00:00
4,13.3,4.7,99,0,0.0,0.0,2022-01-01 04:00:00


In [19]:
merged = price_dataset.merge(solar_dataset, on="datetime", how="inner")
merged = merged.merge(meteo_dataset, on="datetime", how="inner")
merged = merged.sort_values("datetime").reset_index(drop=True)

print(merged.shape)
merged.head()

(35064, 9)


,Price (EUR/MWhe),datetime,solar_generation_MW,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation
0,114.90,2022-01-01 00:00:00,75.0,8.7,9.3,103,0,0.0,0.0
1,113.87,2022-01-01 01:00:00,75.0,10.2,8.6,112,26,0.0,0.0
2,97.80,2022-01-01 02:00:00,75.0,9.0,10.3,115,0,0.0,0.0
3,97.80,2022-01-01 03:00:00,75.0,11.7,7.1,105,0,0.0,0.0
4,95.74,2022-01-01 04:00:00,75.0,13.3,4.7,99,0,0.0,0.0


In [20]:
merged.to_csv("../data/processed/spain_merged.csv", index=False)